In [2]:
import pennylane as qml
import numpy as np
from pennylane import numpy as pnp

############################################
# 설정
############################################
n_qubits = 3
dev = qml.device("default.qubit", wires=n_qubits)

L = 4  # PQC depth
batch_size = 20

############################################
# 1. Target circuit (너 그림 기반으로 근사 구현)
############################################
@qml.qnode(dev)
def target_circuit(state):
    qml.MottonenStatePreparation(state, wires=range(n_qubits))

    qml.RZ(np.pi/8, wires=0)

    # Middle wire (E)
    qml.RX(13*np.pi/8, wires=1)

    # Bottom wire (E*)
    qml.RY(12*np.pi/8, wires=2)
    qml.RX(5*np.pi/8, wires=2)
    qml.RZ(3*np.pi/2, wires=2)
    qml.RX(np.pi/2, wires=2)

    # First entangling block
    qml.CZ(wires=[0,1])

    # Middle wire (E)
    qml.RZ(np.pi, wires=1)
    qml.RY(9*np.pi/8, wires=1)
    qml.RX(np.pi/4, wires=1)

    # Second entangling block
    qml.CZ(wires=[0,1])

    # Second block
    qml.RX(3*np.pi/2, wires=0)
    qml.RZ(np.pi/8, wires=0)
    qml.RY(np.pi/2, wires=1)
    qml.RZ(11*np.pi/8, wires=1)

    # Third entangling block
    qml.CZ(wires=[1,2])

    # Middle wire (E)
    qml.RY(3*np.pi/2, wires=1)

    # Final entangling block
    qml.CZ(wires=[0,1])

    # Final rotations
    qml.RX(3*np.pi/2, wires=0)
    qml.RY(15*np.pi/8, wires=0)

    return qml.state()


############################################
# 2. PQC (trainable)
############################################
@qml.qnode(dev)
def pqc(params, state):
    qml.MottonenStatePreparation(state, wires=range(n_qubits))

    for l in range(L):

        # single-qubit rotations
        for q in range(n_qubits):
            qml.RX(params[l, q, 0], wires=q)
            qml.RY(params[l, q, 1], wires=q)
            qml.RZ(params[l, q, 2], wires=q)

        # entangling (chain)
        for q in range(n_qubits - 1):
            qml.CNOT(wires=[q, q+1])

    return qml.state()


############################################
# 3. Random state 생성 (Haar random)
############################################
def random_state():
    dim = 2**n_qubits
    real = np.random.randn(dim)
    imag = np.random.randn(dim)
    vec = real + 1j * imag
    vec /= np.linalg.norm(vec)
    return vec


############################################
# 4. Fidelity loss
############################################
def fidelity(a, b):
    return np.abs(np.vdot(a, b))**2


def loss_fn(params):
    loss = 0

    for _ in range(batch_size):
        psi = random_state()

        target = target_circuit(psi)
        pred   = pqc(params, psi)

        loss += (1 - fidelity(target, pred))

    return loss / batch_size


############################################
# 5. 학습
############################################
params = pnp.random.normal(scale=0.1, size=(L, n_qubits, 3), requires_grad=True)

opt = qml.AdamOptimizer(stepsize=0.01)

steps = 300

for step in range(steps):
    params = opt.step(loss_fn, params)

    if step % 20 == 0:
        print(f"Step {step}: Loss = {loss_fn(params):.6f}")


############################################
# 6. 최종 평가
############################################
def evaluate(num_test=50):
    scores = []
    for _ in range(num_test):
        psi = random_state()
        target = target_circuit(psi)
        pred   = pqc(params, psi)
        scores.append(fidelity(target, pred))

    print(f"\nAverage fidelity: {np.mean(scores):.6f}")

evaluate()

Step 0: Loss = 0.827539
Step 20: Loss = 0.808915
Step 40: Loss = 0.703328
Step 60: Loss = 0.670594
Step 80: Loss = 0.709880
Step 100: Loss = 0.706360
Step 120: Loss = 0.653477
Step 140: Loss = 0.585050
Step 160: Loss = 0.498412
Step 180: Loss = 0.468166
Step 200: Loss = 0.487170
Step 220: Loss = 0.494799
Step 240: Loss = 0.443428
Step 260: Loss = 0.451509
Step 280: Loss = 0.453153

Average fidelity: 0.545943
